In [ ]:
import os
import shlex
import subprocess

WORKSHOP_RESOURCE_GROUP = os.getenv("WORKSHOP_RESOURCE_GROUP", "rg-delete-me-01").strip()
WORKSHOP_AUTH_MODE = os.getenv("WORKSHOP_AUTH_MODE", "managed-identity").strip()
os.environ["WORKSHOP_RESOURCE_GROUP"] = WORKSHOP_RESOURCE_GROUP
os.environ["WORKSHOP_AUTH_MODE"] = WORKSHOP_AUTH_MODE
print(f"Notebook config: WORKSHOP_RESOURCE_GROUP={WORKSHOP_RESOURCE_GROUP}, WORKSHOP_AUTH_MODE={WORKSHOP_AUTH_MODE}")

cmd = [
    "bash",
    "../../scripts/assign-workshop-env.sh",
    "--resource-group",
    WORKSHOP_RESOURCE_GROUP,
    "--auth-mode",
    WORKSHOP_AUTH_MODE,
]

result = subprocess.run(cmd, check=True, capture_output=True, text=True)
if result.stderr.strip():
    print(result.stderr.strip())

for line in result.stdout.splitlines():
    line = line.strip()
    if not line.startswith("export "):
        continue
    key, raw_value = line[len("export "):].split("=", 1)
    parsed = shlex.split(raw_value)
    os.environ[key] = parsed[0] if parsed else ""

print("Workshop environment variables loaded into notebook kernel. Azure token auth uses AzureCliCredential via az login.")

In [ ]:
import importlib
import sys
from pathlib import Path

NOTEBOOK_PATH_CANDIDATES = [Path.cwd(), Path.cwd() / "AgentWorkshop" / "Notebook"]
for candidate in NOTEBOOK_PATH_CANDIDATES:
    if (candidate / "workshop_bootstrap.py").exists():
        resolved_candidate = str(candidate.resolve())
        if resolved_candidate not in sys.path:
            sys.path.insert(0, resolved_candidate)

import workshop_bootstrap
importlib.reload(workshop_bootstrap)
build_workshop_config = workshop_bootstrap.build_workshop_config

CONFIG_OVERRIDES = {
    "resource_group_name": "",
    "location": "",
    "subscription_id": "",
    "foundry_account_name": "",
    "foundry_project_name": "",
    "foundry_project_endpoint": "",
    "foundry_project_api_key": "",
    "search_service_name": "",
    "search_api_key": "",
    "storage_account_name": "",
    "application_insights_name": "",
    "model_zone": "",
}

config = build_workshop_config(CONFIG_OVERRIDES)
config.show()

# Workshop 0: Baseline Validation

This notebook mirrors docs/index.md and validates the baseline resources after running scripts/deploy-coffee-workshop.sh.

It checks:
- active subscription and resource group
- Microsoft.OperationalInsights registration
- deployed resources in the resource group
- data containers uploaded to storage

In [ ]:
from workshop_bootstrap import run_az

active_account = run_az(["account", "show"], expect_json=True)
print("Active tenant:   ", active_account.get("tenantId", ""))
print("Active account:  ", active_account.get("user", {}).get("name", ""))
print("Subscription id: ", active_account.get("id", ""))

if config.subscription_id and active_account.get("id") != config.subscription_id:
    run_az(["account", "set", "--subscription", config.subscription_id])
    print("Updated active subscription to configured AZURE_SUBSCRIPTION_ID.")

group = run_az(["group", "show", "--name", config.resource_group_name], expect_json=True)
print("Resource group found:", group.get("name"))
print("Region:             ", group.get("location"))

In [ ]:
import os
import subprocess
import time
from workshop_bootstrap import WorkshopConstants

PROVIDER_NAMESPACE = "Microsoft.OperationalInsights"
EXPECTED_BLOB_CONTAINERS = ("coffeecsv", "coffeerecipes", "healtheffects")
BLOB_SAMPLE_LIMIT = 1

run_az(["provider", "register", "--namespace", PROVIDER_NAMESPACE])
for attempt in range(WorkshopConstants.DEFAULT_MAX_POLLS):
    state = run_az(["provider", "show", "--namespace", PROVIDER_NAMESPACE, "--query", "registrationState", "--output", "tsv"])
    print(f"Provider state attempt {attempt + 1}: {state}")
    if state == "Registered":
        break
    time.sleep(WorkshopConstants.DEFAULT_POLL_SECONDS)
else:
    raise TimeoutError("Provider registration did not finish in time.")

resources = run_az(["resource", "list", "--resource-group", config.resource_group_name], expect_json=True)
print("\nDiscovered resources:")
for resource in sorted(resources, key=lambda item: item.get("type", "")):
    print(f"- {resource.get('type', '')}: {resource.get('name', '')}")

auth_mode = os.getenv("WORKSHOP_AUTH_MODE", "key").strip().lower()
storage_key = ""
if auth_mode == "key":
    try:
        storage_key = run_az([
            "storage",
            "account",
            "keys",
            "list",
            "--resource-group",
            config.resource_group_name,
            "--account-name",
            config.storage_account_name,
            "--query",
            "[0].value",
            "--output",
            "tsv",
        ])
    except subprocess.CalledProcessError as exc:
        message = (exc.stderr or str(exc)).strip()
        print(f"Storage key lookup skipped: {message}")
else:
    print("Managed identity mode detected: using login auth for blob checks.")

print("\nContainer validation:")
for container_name in EXPECTED_BLOB_CONTAINERS:
    blob_args = [
        "storage",
        "blob",
        "list",
        "--account-name",
        config.storage_account_name,
        "--container-name",
        container_name,
        "--num-results",
        str(BLOB_SAMPLE_LIMIT),
        "--query",
        "length(@)",
        "--output",
        "tsv",
    ]
    if storage_key:
        blob_args.extend(["--account-key", storage_key])
    else:
        blob_args.extend(["--auth-mode", "login"])

    try:
        count = run_az(blob_args)
        print(f"- {container_name}: {count} blob(s) found in sample query")
    except subprocess.CalledProcessError as exc:
        message = (exc.stderr or str(exc)).strip()
        print(f"- {container_name}: unable to validate blobs ({message})")

## Next Workshop Notebooks

1. 02_foundry_model_deployments.ipynb
2. 03_knowledge_base_foundry_iq.ipynb
3. 04_agents_workshop.ipynb
4. 05_multi_agent_workshop.ipynb
5. 06_multi_agent_2_workshop.ipynb